# T6 — Optimizer interaction (`S7_optimizers`)

Task T6 from the `Aditya` section of the repo README. **No new code**: the study is already
defined in `src/studies.py:210`, and `sgd` / `sgd_momentum` / `adam` all already exist in
`src/train.py:177-185`. This notebook launches it, survives the night, and reports it.

**Question.** Does the effect survive momentum, and does it survive adaptive preconditioning?
The paper's decomposition is stated for gradient flow; a diagonal preconditioner rescales
coordinates independently and does not preserve the radial/tangential split in activation
space. Predicted: **survives momentum, weakens under Adam.**

**Arms.** 3 optimizers x {baseline λ=0, penalty λ*=0.003} x seeds {1,2,3} = **18 runs**.
Learning rate is per optimizer — sgd 0.1, sgd_momentum 0.01, adam 0.001 — and **must be
stated in the final table**. Round 1-4 compared optimizers across two code generations at an
undisclosed 10x learning-rate difference and had to withdraw the result.

## Runtime budget on a Kaggle T4

Grounded in the archive: **79 completed runs** at this exact configuration (150 tasks x 10
epochs, width 1000, depth 3) have a median wall-clock of **63.6 min** (mean 68.7, range
35-109) on the authors' lab GPU at 8-way concurrency. Two corrections for your setting,
pulling opposite ways:

* **Cheaper:** S7 leaves `track_drift` off; all 79 archived runs had it on, so you skip the
  entire drift battery.
* **Dearer:** a T4 is slower than their card, and Kaggle gives ~4 CPU cores against the lab
  machine's 192 — the spectral metrics (`eff_rank`, `subspace_overlap`) run on CPU in float64.

Planning number: **~90 min per run** (75-120).

| | runs | waves at concurrency 2 | wall-clock |
|---|---|---|---|
| one shard (e.g. `"sgd"`) | 6 | 3 | **~4.5 h** (3.5-6 h) |
| all three shards | 18 | 9 | ~13.5 h |

One shard per session sits well inside the 12 h cap. Two sessions can run **concurrently**
(Kaggle allows 2 GPU sessions), so: `"sgd"` and `"sgd_momentum"` in parallel, then `"adam"`
with both outputs attached — T6 done in ~9 h of calendar time. Quota burns at 2x while two
sessions run; the 30 h/week allowance covers T6 (~13.5 h) and T10 (~9 h) together.

## Overnight checklist

1. **Internet ON** (Settings -> Internet) — needed to clone and to download MNIST.
2. **Accelerator: GPU T4 x2.**
3. **Save & Run All (Commit)**, not interactive. An interactive session idles out after
   ~20-40 min of inactivity and you lose the run; a commit executes headless and saves its
   output at the end.
4. **Sessions 2+:** Add Data -> Your Notebooks -> Output, attach every earlier session.

## What makes it failsafe

* **A hard time budget.** The launcher is stopped gracefully at `TIME_BUDGET_H` so the
  remaining cells still run and `/kaggle/working` is still saved. A session killed at the
  12 h wall saves nothing — this is the single most important guard, and it is why the
  budget defaults to 10.5 h.
* **Resumable.** Every completed run is skipped on the next session; a run interrupted
  mid-flight is re-run from scratch (its `config.json` is left at `status: running`, which
  the launcher treats as failed).
* **Nothing after the launch raises.** Analysis cells catch their own exceptions, so one bad
  cell cannot abort the notebook and cost you the output.
* **`SHARD = "auto"`** picks the optimizer with the most unfinished runs, so a session
  started at 3 a.m. does the most useful available work without you choosing.

## 1. Configuration

In [ ]:
# Which arms to run this session.
#   "auto"  -> the optimizer with the most unfinished runs (resolved in cell 6)
#   "sgd" | "sgd_momentum" | "adam" | "all"  -> explicit
SHARD = "all"

# Stop launching and shut down cleanly after this many hours, so the notebook finishes and
# /kaggle/working is saved. Kaggle kills a session at 12 h and saves NOTHING, so leave margin.
TIME_BUDGET_H = 11.0

# Kaggle T4 x2: 2 GPUs, ~4 CPU cores. CONCURRENCY * THREADS must stay <= nproc, because the
# metric SVDs are CPU-bound (see limit_threads() in src/train.py).
CONCURRENCY = 2
THREADS     = 2
GPUS        = "0,1"

REPO = "https://github.com/frisco137/rs_for_optimisation.git"
WORK = "/kaggle/working/rs_for_optimisation"
STUDY = "S7_optimizers"
ROOT  = "experiments/permuted_mnist/optimizers"

print(f"shard={SHARD}  budget={TIME_BUDGET_H}h  concurrency={CONCURRENCY}x{THREADS} threads")

## 2. Environment

Fatal on purpose: a run without a GPU would take days and produce nothing usable, and a
`CONCURRENCY x THREADS` above `nproc` makes the CPU-bound metric SVDs thrash — Round 1-4 saw
load average 484 and tasks failing to advance for four minutes.

In [ ]:
import multiprocessing, os, shutil, subprocess, sys, time

SESSION_START = time.time()

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)
ncpu = multiprocessing.cpu_count()
print("CPU cores:", ncpu, "| free disk GB:", round(shutil.disk_usage("/kaggle/working").free / 1e9, 1))
assert CONCURRENCY * THREADS <= ncpu, f"{CONCURRENCY}x{THREADS} exceeds {ncpu} cores"

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
assert torch.cuda.is_available(), "Turn on the GPU accelerator (Settings -> Accelerator)."

# The GPU must actually be usable by THIS torch build. Kaggle hands out a P100 (sm_60) when
# the accelerator is left as plain "GPU", and its PyTorch ships no Pascal kernels -- every run
# then dies on its first CUDA op with "no kernel image is available for execution on the
# device", after the CPU-only checks above have all passed. Fail here, in seconds, instead.
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f"device: {name}  sm_{cap[0]}{cap[1]}  | torch arches: {torch.cuda.get_arch_list()}")
try:
    _t = torch.zeros(64, 64, device="cuda")
    _ = (_t @ _t).sum().item()
    torch.cuda.synchronize()
except Exception as e:
    raise SystemExit(
        f"GPU UNUSABLE: {name} (sm_{cap[0]}{cap[1]}) with this torch build -- {e}\n"
        f"Set the accelerator to T4 x2: kernel-metadata.json \"machine_shape\": \"NvidiaTeslaT4\", "
        f"or Settings -> Accelerator -> GPU T4 x2 in the editor. Do not launch on a P100.")
print("GPU compute check passed")
del _t

## 3. Clone and install

In [ ]:
import os, shutil, subprocess, sys, time

# A transient git failure must not end the session: one observed clone returned exit 128 with
# a healthy network moments after an identical clone succeeded. Retry with backoff, and clear
# any half-written tree between attempts.
for attempt in range(1, 6):
    if os.path.exists(os.path.join(WORK, "src", "train.py")):
        break
    shutil.rmtree(WORK, ignore_errors=True)
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, WORK],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"cloned on attempt {attempt}")
        break
    print(f"clone attempt {attempt} failed (exit {r.returncode}): {r.stderr.strip()[-200:]}")
    time.sleep(15 * attempt)
else:
    raise SystemExit("CLONE FAILED after 5 attempts -- not launching.")

os.chdir(WORK)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!pip install -q -r requirements.txt

## 4. Preflight

`src/data/` — the package holding `PermutedMNIST` — was missing from the repository until
commit `bff5bd9`: `.gitignore` had `*/data/`, meant for the downloaded MNIST in `data/`, and
the pattern also matched `src/data/`. `src/train.py:35-36` imports it, so every run died at
import while the 30 tests still passed and `--dry-run` still printed its commands. It is
tracked now; this cell refuses to launch if that ever regresses.

`torchvision` is a hard dependency of the loader and is **not** in `requirements.txt`. Kaggle
preinstalls it; the cell installs it if absent rather than letting 18 runs die on start.

The loader check confirms the property the paired analysis depends on: two arms built with the
same seed get the same 150 permutations, so `baseline/seed_1` and `penalty/seed_1` differ only
by the penalty.

In [ ]:
import os, subprocess, sys

assert os.path.exists("src/data/permuted_mnist.py"), (
    "src/data/ is missing -- the .gitignore `*/data/` bug is back. "
    "Pull latest main (fixed in bff5bd9); do not launch without it.")

try:
    import torchvision
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchvision"], check=True)
    import torchvision
print("torchvision", torchvision.__version__)

chk = subprocess.run([sys.executable, "-c", "import src.train"],
                     env={**os.environ, "PYTHONPATH": "."}, capture_output=True, text=True)
if chk.returncode != 0:
    print(chk.stderr.strip()[-800:])
    raise SystemExit("PREFLIGHT FAILED -- src.train does not import; do not launch.")
print("preflight OK -- src.train imports")

sys.path.insert(0, ".")
import torch
from src.data.permuted_mnist import PermutedMNIST

a = PermutedMNIST(n_tasks=150, device="cpu", seed=1)
b = PermutedMNIST(n_tasks=150, device="cpu", seed=1)
c = PermutedMNIST(n_tasks=150, device="cpu", seed=2)
assert all(torch.equal(x, y) for x, y in zip(a.permutations, b.permutations)), \
    "same seed must give the same task sequence -- paired arms would not be paired"
assert not torch.equal(a.permutations[5], c.permutations[5]), "different seeds must differ"
assert len({tuple(p.tolist()) for p in a.permutations}) == 150, "permutations must be distinct"
print("loader OK -- seed 1 reproducible, seeds independent, 150 distinct permutations")

x = a.x_train
print(f"pixel mean {x.mean():+.4f}  std {x.std():.4f}   (standardised: 0.0 / 1.0)")
print(f"||x|| mean {x.norm(dim=1).mean():.2f}          (expected ~27.7)")
del a, b, c, x

## 5. Carry over earlier sessions

Runs finished in an earlier session are copied back from every attached input dataset. The
launcher reads each run's `config.json`: `complete` under a matching configuration is skipped,
`running` (a session killed mid-flight) is treated as failed and re-run, and a run under a
configuration that no longer matches is `stale` and re-run — a study cannot silently mix
configurations.

In [ ]:
import glob, os, shutil

restored = 0
for src in glob.glob(f"/kaggle/input/*/rs_for_optimisation/{ROOT}"):
    for run in glob.glob(os.path.join(src, "*", "*", "seed_*")):
        dst = os.path.join(WORK, ROOT, os.path.relpath(run, src))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copytree(run, dst)
            restored += 1
print(f"restored {restored} run dir(s) from {len(glob.glob('/kaggle/input/*'))} attached input(s)")

## 6. Resolve the shard

`--only` matches substrings against each run directory (`src/launch.py:116`), and arm dirs are
`<optimizer>/<baseline|penalty>/seed_N`, so an optimizer name selects its 6 runs.

`SHARD = "auto"` counts what is already complete and picks the optimizer with the most work
left, so an unattended session always does something useful and two parallel sessions started
from the same notebook do not collide on the same arms — check the printed choice if you are
running two at once.

In [ ]:
import glob, json, os

OPTS = ["sgd", "sgd_momentum", "adam"]

def completed(opt):
    n = 0
    for c in glob.glob(os.path.join(ROOT, opt, "*", "seed_*", "config.json")):
        try:
            if json.load(open(c)).get("status") == "complete":
                n += 1
        except Exception:
            pass
    return n

counts = {o: completed(o) for o in OPTS}
print("completed per optimizer (of 6 each):", counts)

if SHARD == "auto":
    remaining = {o: 6 - n for o, n in counts.items() if n < 6}
    if not remaining:
        SHARD_RESOLVED, ONLY = None, None
        print("\nAll 18 runs are complete -- nothing to launch. Skip to the analysis cells.")
    else:
        SHARD_RESOLVED = max(remaining, key=lambda o: (remaining[o], -OPTS.index(o)))
        ONLY = f"{SHARD_RESOLVED}/"
        print(f"\nauto -> {SHARD_RESOLVED}  ({remaining[SHARD_RESOLVED]} runs left)")
elif SHARD == "all":
    SHARD_RESOLVED, ONLY = "all", None
    print("\nrunning ALL 18 -- expect ~13.5 h, which will hit the time budget")
else:
    SHARD_RESOLVED = SHARD
    ONLY = ",".join(s.strip() + "/" for s in SHARD.split(","))
    print(f"\nexplicit -> {SHARD_RESOLVED}  (--only {ONLY})")

## 7. Regression tests

30 tests, all should pass. Two exist because of bugs that invalidated Rounds 1-4 and both are
live hazards here: `test_phi_rad_tilde_leakage` (the metric must use the task-loss gradient
only, never the total gradient, or it partly measures the regulariser) and
`test_clip_after_scale_destroys_the_match_THE_ROUND_1_4_BUG` (anything rescaling gradients
must run after `clip_grad_norm_`, or the clip renormalises it away and the intervention
silently does nothing). Do not launch on a red suite.

In [ ]:
!PYTHONPATH=. python3 -m pytest tests/ -q

## 8. Dry run

Prints the planned runs and their arguments without training. Check the per-optimizer learning
rates here — **sgd 0.1, sgd_momentum 0.01, adam 0.001**.

In [ ]:
!PYTHONPATH=. python3 -m src.launch --study {STUDY} --dry-run

## 9. Launch — the overnight cell

Guards in force:

* **Time budget.** At `TIME_BUDGET_H` the launcher gets SIGTERM, then SIGKILL 60 s later. The
  notebook continues to the analysis cells so `/kaggle/working` is saved. Runs in flight are
  lost and re-run next session; runs already finished are safe.
* **Heartbeat.** A progress line every 10 minutes with elapsed time and completed count, so a
  stalled session is visible in the log rather than looking like a slow one.
* **Output throttling.** tqdm redraws are dropped; only real events are printed. A 4.5 h run
  otherwise produces tens of MB of progress bars.
* **Retries.** The launcher re-attempts a failed run twice (`--retries 2`) — a shared GPU can
  OOM-kill a run through no fault of its own — and records failures in `_launch_report.json`.
* **No exception escapes.** A crash here is caught and reported so the later cells still run.

In [ ]:
import glob, json, os, signal, subprocess, sys, threading, time

def n_complete():
    n = 0
    for c in glob.glob(os.path.join(ROOT, "*", "*", "seed_*", "config.json")):
        try:
            if json.load(open(c)).get("status") == "complete":
                n += 1
        except Exception:
            pass
    return n

if SHARD_RESOLVED is None:
    print("nothing to launch")
else:
    cmd = [sys.executable, "-u", "-m", "src.launch", "--study", STUDY,
           "--gpus", GPUS, "--concurrency", str(CONCURRENCY),
           "--threads", str(THREADS), "--retries", "2"]
    if ONLY:
        cmd += ["--only", ONLY]
    print(" ".join(cmd), flush=True)

    deadline = SESSION_START + TIME_BUDGET_H * 3600
    env = {**os.environ, "PYTHONPATH": ".", "PYTHONUNBUFFERED": "1"}
    t0 = time.time()
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, preexec_fn=os.setsid)

    state = {"stop": False}

    def watchdog():
        """Enforce the deadline and beat independently of the child's output.

        The read loop below blocks on p.stdout, so a run that goes quiet -- a hang,
        a wedged GPU -- would never reach a deadline check placed in that loop.
        This thread owns both the budget and the heartbeat for that reason.
        """
        last_beat = time.time()
        while not state["stop"]:
            time.sleep(5)
            now = time.time()
            if now - last_beat >= 600:
                print(f"    [heartbeat] {(now - t0) / 3600:.2f} h elapsed, "
                      f"{n_complete()}/18 complete", flush=True)
                last_beat = now
            if now > deadline and p.poll() is None:
                print(f"\n!! TIME BUDGET {TIME_BUDGET_H} h REACHED -- stopping the launcher "
                      f"so this session's finished runs are saved.", flush=True)
                state["stop"] = True
                try:
                    os.killpg(os.getpgid(p.pid), signal.SIGTERM)
                    time.sleep(60)
                    if p.poll() is None:
                        os.killpg(os.getpgid(p.pid), signal.SIGKILL)
                except ProcessLookupError:
                    pass
                return

    wd = threading.Thread(target=watchdog, daemon=True)
    wd.start()
    try:
        for line in p.stdout:
            s = line.rstrip("\n")
            # tqdm redraws would flood the log; drop them, keep real events
            if "\r" in line or ("%|" in s and s.strip().endswith("]")):
                continue
            if s.strip():
                print(s, flush=True)
    except Exception as e:
        print("launch loop error:", repr(e))
    finally:
        state["stop"] = True
    try:
        p.wait(timeout=180)
    except Exception:
        try:
            os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except Exception:
            pass
    print(f"\nexit {p.returncode} after {(time.time() - t0) / 3600:.2f} h; "
          f"{n_complete()}/18 runs complete")

## 10. What completed

An arm with missing seeds is **not reportable** — `analysis/common.py` raises rather than
averaging the survivors, which is how a published table once came to contain a single-seed
mean with a `nan` standard deviation.

In [ ]:
import glob, json, os
import pandas as pd

try:
    rows = []
    for c in sorted(glob.glob(os.path.join(ROOT, "*", "*", "seed_*", "config.json"))):
        cfg = json.load(open(c))
        rows.append({"arm": "/".join(c.split("/")[-4:-2]), "seed": cfg.get("seed"),
                     "status": cfg.get("status", "?"), "optimizer": cfg.get("optimizer"),
                     "lr": cfg.get("lr"), "lambda_rs": cfg.get("lambda_rs"),
                     "hours": round(cfg.get("wall_clock_sec", 0) / 3600, 2),
                     "config_hash": cfg.get("config_hash")})
    df = pd.DataFrame(rows)
    if len(df):
        print(df.sort_values(["arm", "seed"]).to_string(index=False))
        print("\ncomplete:", int((df.status == "complete").sum()), "of 18")
        bad = df[df.status != "complete"]
        if len(bad):
            print("not complete:", bad[["arm", "seed", "status"]].to_string(index=False))
    else:
        print("no runs yet")
    rep = os.path.join(ROOT, "_launch_report.json")
    if os.path.exists(rep):
        print("\n_launch_report.json:", json.dumps(json.load(open(rep)))[:1500])
except Exception as e:
    print("status cell error (non-fatal):", repr(e))

## 11. The identical-arms check

**Read this before reporting anything.** Round 1-4's two adaptive arms turned out to be
bitwise identical because both had weight decay zero, so `AdamW(0) == Adam(0)`; the repo's
conventions list it as rule 5, *identical values across arms to four decimals are a bug
signal*. This study runs only one adaptive optimizer for that reason — but the same signature
appears if the penalty silently does nothing in an arm, which is exactly what T6 is testing
for under Adam. Two things are checked: final metrics agreeing to 4 dp, and `config_hash`
collisions between arms that are supposed to differ.

In [ ]:
import glob, itertools, json, os
import numpy as np
import pandas as pd

try:
    fin = {}
    for m in sorted(glob.glob(os.path.join(ROOT, "*", "*", "seed_*", "metrics.parquet"))):
        arm, seed = "/".join(m.split("/")[-4:-2]), m.split("/")[-2]
        d = pd.read_parquet(m)
        last = d[d.task >= d.task.max() - 19]          # the reported final-20-task window
        fin[(arm, seed)] = {k: float(last[k].mean())
                            for k in ["test_acc", "prev_only_acc", "radius_mean"] if k in last}
    if fin:
        t = pd.DataFrame(fin).T
        print(t.round(4).to_string(), "\n")
        dup = 0
        for a, b in itertools.combinations(t.index, 2):
            if a[1] == b[1] and np.allclose(t.loc[[a]].values, t.loc[[b]].values, atol=1e-4):
                print(f"!! IDENTICAL to 4dp: {a} vs {b} -- find out why before reporting")
                dup += 1
        print("no 4dp collisions" if not dup else f"{dup} collision(s)")
        hashes = {}
        for c in glob.glob(os.path.join(ROOT, "*", "*", "seed_*", "config.json")):
            cfg = json.load(open(c))
            hashes.setdefault((cfg.get("config_hash"), cfg.get("seed")), set()).add(
                "/".join(c.split("/")[-4:-2]))
        for k, v in hashes.items():
            if len(v) > 1:
                print("!! config_hash collision across arms:", v)
    else:
        print("no metrics yet")
except Exception as e:
    print("identity-check error (non-fatal):", repr(e))

## 12. Report

`analysis/sweeps.py` pairs every arm against its own control and prints test statistic, p, n
and sign split. It **refuses** arms with missing seeds — intended behaviour, not an error.
`--preliminary` overrides and stamps every row with its true `n`, which is what you want
between sessions.

Report `prev_only_acc`, not `avg_seen_acc`. The latter includes the current task, so it is
inflated by `test_acc/(t+1)` and biased toward arms with higher current-task accuracy —
exactly the confound "at matched current-task accuracy" is meant to remove.

In [ ]:
!PYTHONPATH=. python3 analysis/sweeps.py {STUDY} 2>&1 | tail -60

In [ ]:
# Partial view between sessions: reports incomplete arms with their true n.
!PYTHONPATH=. python3 analysis/sweeps.py {STUDY} --preliminary 2>&1 | tail -60

## 13. Calibration against the archive

Your `sgd/baseline` is the same configuration as the archived
`stiffness_curve/lambda_0.0000`, on the authors' own loader, so task 0 should agree closely.
Verified locally before this notebook shipped: `radius_mean` 45.54 vs 45.58, `eff_rank` 234.70
vs 234.61, `weight_norm` matching to six significant figures. A visible mismatch means
something upstream changed and is worth chasing before you trust the sweep.

In [ ]:
import os
import pandas as pd

try:
    mine = os.path.join(ROOT, "sgd/baseline/seed_1/metrics.parquet")
    ref = "experiments/permuted_mnist/stiffness_curve/lambda_0.0000/seed_1/metrics.parquet"
    if os.path.exists(mine) and os.path.exists(ref):
        cols = ["layer", "radius_mean", "weight_norm", "test_acc", "eff_rank"]
        m, a = pd.read_parquet(mine), pd.read_parquet(ref)
        cmp = m[m.task == 0][cols].merge(a[a.task == 0][cols], on="layer",
                                         suffixes=("_yours", "_archived"))
        print(cmp.round(4).to_string(index=False))
    else:
        print("run sgd/baseline/seed_1 first, then re-run this cell")
except Exception as e:
    print("calibration error (non-fatal):", repr(e))

## 14. Refresh STUDY.md and save the output

Everything under `/kaggle/working` becomes this notebook's output. MNIST is deleted first —
it is re-downloadable and would otherwise bloat every attached dataset.

In [ ]:
import shutil
try:
    !PYTHONPATH=. python3 -m src.docs
    shutil.rmtree(os.path.join(WORK, "data"), ignore_errors=True)
    !du -sh {WORK}/{ROOT} /kaggle/working
    print(open(os.path.join(ROOT, "STUDY.md")).read()[:2500])
except Exception as e:
    print("docs/cleanup error (non-fatal):", repr(e))

## After all three shards

1. Re-run cells 10-13 in the final session, with the earlier outputs attached, so the report
   covers all 18 runs and `sweeps.py` stops refusing arms.
2. Write the finding into `experiments/permuted_mnist/optimizers/STUDY.md` between the
   `<!-- FINDING -->` markers.
3. **State the per-optimizer learning rate in the table** — 0.1 / 0.01 / 0.001. Not a footnote.
4. Report `prev_only_acc`. Per-layer metrics stay per layer, never averaged across layers.
5. An arm that fails its own acceptance test is not reported at all (convention 6).

The claim to answer: does the penalty's retention benefit **survive momentum** and **weaken
under Adam**? The archived SGD reference is `prev_only_acc` 0.207 at λ=0 against 0.342 at
λ*=0.003 — a gap of ~0.135 at matched current-task accuracy. Does that gap persist per row?

If Adam shows the effect undiminished, that **contradicts** the paper's stated mechanism.
Report it as a finding, not a disappointment — and note in `STUDY.md` that λ was held at the
SGD-tuned optimum for all three optimizers, so a weakened Adam effect is consistent with
either mechanism disruption or λ mis-tuning, and separating them needs a per-optimizer sweep.

## Trim the output

`/kaggle/working` becomes this notebook's output dataset, and it currently holds the entire
cloned repo — ~98 MB, almost all of it the archived `experiments/` tree that came from git and
is identical in every session. Attaching that to the next session is slow for no benefit.

This keeps only what the next session actually needs: **this study's run directories** (their
`config.json`, `metrics.parquet`, checkpoints and `selectivity.npz`), the study's `STUDY.md` and
`_launch_report.json`, and any generated figures. Everything else is re-cloned from git next
time.

It runs **last**, after `src.docs`, and never deletes the working directory itself. The keep
list is printed before anything is removed, and a guard refuses to run if the study directory
is missing — so a bug here cannot silently throw away a night of compute.

In [ ]:
import os, shutil

try:
    root = "/kaggle/working"
    repo = os.path.join(root, "rs_for_optimisation")
    keep_rel = [ROOT, "latex/figs"]                    # study runs, and any generated figures

    keep_abs = [os.path.join(repo, r) for r in keep_rel]
    study_dir = os.path.join(repo, ROOT)
    if not os.path.isdir(study_dir):
        print(f"REFUSING to trim: {study_dir} does not exist")
    else:
        n_runs = sum(1 for _, ds, fs in os.walk(study_dir) if "config.json" in fs)
        before = sum(os.path.getsize(os.path.join(dp, f))
                     for dp, _, fs in os.walk(root) for f in fs
                     if os.path.exists(os.path.join(dp, f)))
        print(f"keeping {n_runs} run dir(s) under {ROOT}")

        # Everything under the repo that is not on a kept path, plus stray top-level items.
        def protected(path):
            return any(path == k or path.startswith(k + os.sep) or k.startswith(path + os.sep)
                       for k in keep_abs)

        removed = 0
        for dirpath, dirnames, filenames in os.walk(repo, topdown=True):
            if protected(dirpath):
                if dirpath in keep_abs:
                    dirnames[:] = []                   # keep this subtree whole
                continue
            for d in list(dirnames):
                full = os.path.join(dirpath, d)
                if not protected(full):
                    shutil.rmtree(full, ignore_errors=True)
                    dirnames.remove(d)
                    removed += 1
            for f in filenames:
                full = os.path.join(dirpath, f)
                if not protected(full):
                    try:
                        os.remove(full); removed += 1
                    except OSError:
                        pass

        after = sum(os.path.getsize(os.path.join(dp, f))
                    for dp, _, fs in os.walk(root) for f in fs
                    if os.path.exists(os.path.join(dp, f)))
        kept_runs = sum(1 for _, ds, fs in os.walk(study_dir) if "config.json" in fs)
        print(f"removed {removed} item(s): {before/1e6:.1f} MB -> {after/1e6:.1f} MB")
        print(f"run dirs still present: {kept_runs} (was {n_runs})")
        assert kept_runs == n_runs, "TRIM BUG: run directories were lost"
        for dp, _, fs in os.walk(root):
            if fs:
                print(" ", os.path.relpath(dp, root), f"({len(fs)} files)")
except Exception as e:
    print("trim error (non-fatal):", repr(e))